In [1]:
import os
import sys

PROJECT_ROOT = os.path.abspath("..")
sys.path.append(PROJECT_ROOT)

%load_ext autoreload
%autoreload 2

In [2]:
import torch
import torch.nn as nn

import pandas as pd

import numpy as np
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from torch.utils.data import TensorDataset, DataLoader

from src.preprocessing import get_preprocessor

In [3]:
train = pd.read_csv('../data/train.csv')

X = train.drop(columns='SalePrice')
y = train['SalePrice']

y_log = np.log1p(y)

In [13]:
class HpMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )
        
    def forward(self, x):
        return self.net(x).squeeze(1)
        

In [20]:
class THpMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(1)

In [21]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

N_SPLITS = 5
EPOCHS = 100
BATCH_SIZE = 64
LR = 1e-3
SEED = 42

cv = KFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED
)

fold_rmse = []

for fold, (train_idx, val_idx) in enumerate(cv.split(X), start=1):
    print('Fold', fold)
    
    X_train = X.iloc[train_idx]
    X_val = X.iloc[val_idx]
    
    y_train = y_log.iloc[train_idx]
    y_val = y_log.iloc[val_idx]
    
    preprocessor = get_preprocessor(
        X_train,
        model_type='default'
    )
    
    X_train_processed = preprocessor.fit_transform(
        X_train
    )
    
    X_val_precessed = preprocessor.transform(
        X_val
    )
    
    if hasattr(X_train_processed, 'toarray'):
        X_train_processed = X_train_processed.toarray()
        X_val_precessed = X_val_precessed.toarray()
        
    X_train_tensor = torch.tensor(X_train_processed, dtype=torch.float32)
    X_val_tensor = torch.tensor(X_val_precessed, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32)
    y_val_tensor = torch.tensor(y_val.values, dtype=torch.float32)
    
    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    
    model = THpMLP(
        input_dim=X_train_tensor.shape[1]
    ).to(device)
    
    mse = nn.MSELoss()
    
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LR,
    )
    
    for epoch in range(EPOCHS):
        model.train()
        
        for x_batch, y_batch in train_loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)
            
            optimizer.zero_grad()
            
            preds = model(x_batch)
            
            loss = mse(preds, y_batch)
            
            loss.backward()
            optimizer.step()
            
    model.eval()
    
    with torch.no_grad():
        val_preds = model(X_val_tensor.to(device))
        
        val_mse = mse(val_preds, y_val_tensor.to(device))
        
        val_rmse = torch.sqrt(val_mse).item()
        
    fold_rmse.append(val_rmse)
    
    print(f"RMSE: {val_rmse:.6f}")
    
fold_rmse = np.array(fold_rmse)

print("\nFold RMSE:", fold_rmse)
print("Mean RMSE:", fold_rmse.mean())
print("Std RMSE:", fold_rmse.std())

Fold 1
RMSE: 0.135117
Fold 2
RMSE: 0.155803
Fold 3
RMSE: 0.284146
Fold 4
RMSE: 0.156973
Fold 5
RMSE: 0.138113

Fold RMSE: [0.13511701 0.15580316 0.28414589 0.15697263 0.13811322]
Mean RMSE: 0.17403038144111632
Std RMSE: 0.05577260509284286


# Embeddings

In [4]:
cat_features = X.select_dtypes(include=['object', 'string']).columns.tolist()

num_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

cat_features.append('MSSubClass')
num_features.remove('MSSubClass')

In [5]:
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

cat_preprocessor = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='Missing')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

num_preprocessor = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

X_cat = cat_preprocessor.fit_transform(X[cat_features])

X_num = num_preprocessor.fit_transform(X[num_features])

print(X_cat.shape)
print(X_cat[:3])

X_cat = X_cat.astype(np.int64) + 1

encoder = cat_preprocessor.named_steps['encoder']

cardinalities = [
    len(categories) + 1
    for categories in encoder.categories_
]

print('len of cat_features:', len(cat_features))
print('len of cadinalities:', len(cardinalities))
print('first 10 cardinalities:', cardinalities[:10])

embeddings = nn.ModuleList([
    nn.Embedding(
        num_embeddings=cardinality,
        embedding_dim=min(8, cardinality),
    )
    for cardinality in cardinalities
])

(1460, 44)
[[ 3.  1.  1.  3.  3.  0.  4.  0.  5.  2.  2.  0.  5.  1.  1. 12. 13.  1.
   2.  4.  2.  2.  4.  4.  2.  6.  1.  0.  1.  5.  2.  6.  3.  1.  2.  5.
   5.  2.  3.  2.  1.  8.  4.  5.]
 [ 3.  1.  1.  3.  3.  0.  2.  0. 24.  1.  2.  0.  2.  1.  1.  8.  8.  2.
   3.  4.  1.  2.  4.  1.  0.  6.  1.  0.  1.  5.  3.  6.  5.  1.  2.  5.
   5.  2.  3.  2.  1.  8.  4.  0.]
 [ 3.  1.  1.  0.  3.  0.  4.  0.  5.  2.  2.  0.  5.  1.  1. 12. 13.  1.
   2.  4.  2.  2.  4.  3.  2.  6.  1.  0.  1.  5.  2.  6.  5.  1.  2.  5.
   5.  2.  3.  2.  1.  8.  4.  5.]]
len of cat_features: 44
len of cadinalities: 44
first 10 cardinalities: [6, 3, 4, 5, 5, 3, 6, 4, 26, 10]


In [6]:
house_0 = torch.tensor(
    X_cat[0],
    dtype=torch.long,
)

embedded_features = []

for i, embedding in enumerate(embeddings):
    feature_id = house_0[i]
    embedded = embedding(feature_id)
    
    embedded_features.append(embedded)
    
for i, embedded in enumerate(embedded_features[:5]):
    print(i, embedded.shape)
    
house_cat_embedded = torch.cat(
    embedded_features,
    dim=0,
)

print(house_cat_embedded.shape)
print(house_cat_embedded)

house_0_num = torch.tensor(
    X_num[0],
    dtype=torch.float32,
)

print(house_0_num.shape)

house_0_full = torch.cat(
    [
        house_0_num,
        house_cat_embedded,
    ],
    dim=0,
)

print(house_0_full.shape)

0 torch.Size([6])
1 torch.Size([3])
2 torch.Size([4])
3 torch.Size([5])
4 torch.Size([5])
torch.Size([276])
tensor([ 1.8531,  0.6400, -2.4854,  0.7870, -1.9655, -0.4044,  2.6925,  2.7166,
        -2.5507, -0.8384, -1.5681,  1.0988,  1.0813,  0.2865, -0.3154, -0.6230,
        -0.2567, -0.4333,  0.4482,  0.4310,  0.4200,  0.2930, -0.6860,  0.4052,
        -1.6639, -0.4694,  1.1818, -0.9291, -1.1297,  1.0358,  0.5810, -0.0284,
        -1.1091,  0.0754,  0.1653,  1.7677,  0.1422,  0.4805, -0.7965, -1.7192,
        -0.0411, -0.2081, -0.1167,  0.1099,  0.6114,  0.2896,  2.2195, -0.1452,
        -0.6339,  0.4389,  0.2179, -0.6944, -1.3721,  1.6926,  1.4373,  0.0807,
        -1.3817, -1.7246, -0.3569, -0.7760, -0.3843, -1.0907,  0.0356, -0.0976,
         0.9500, -0.0201,  0.0583,  0.3924, -0.9533, -2.1840,  0.8906,  0.4367,
        -0.8793,  0.5787, -0.0247, -0.1658,  0.2851,  0.5493,  1.8901, -0.6391,
        -0.9478, -0.2045, -1.6964,  0.6565, -0.7205,  0.3143, -0.1399,  1.1952,
        -0.3

In [7]:
class HpEmbeddingMLP(nn.Module):
    def __init__(self, num_numeric, cardinalities):
        super().__init__()
        
        self.embeddings = nn.ModuleList([
            nn.Embedding(
                num_embeddings=cardinality,
                embedding_dim=min(8, cardinality),
            )
            for cardinality in cardinalities
        ])
        
        total_embedding_dim = sum(
            embedding.embedding_dim
            for embedding in self.embeddings
        )
        
        input_dim = num_numeric + total_embedding_dim
        
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )
        
    def forward(self, x_num, x_cat):
        embedded_features = []
        
        for i, embedding in enumerate(self.embeddings):
            embedded = embedding(x_cat[:, i])
            embedded_features.append(embedded)
            
        x_embedded = torch.cat(
            embedded_features,
            dim=1,
        )
        
        x = torch.cat(
            [x_num, x_embedded],
            dim=1,
        )
        
        return self.mlp(x).squeeze(1)

In [8]:
x_cat_batch = torch.tensor(
    X_cat[:4],
    dtype=torch.long
)

x_num_batch = torch.tensor(
    X_num[:4],
    dtype=torch.float32
)

model = HpEmbeddingMLP(
    num_numeric=X_num.shape[1],
    cardinalities=cardinalities
)

with torch.no_grad():
    preds = model(
        x_num_batch,
        x_cat_batch
    )
    
print(preds.shape)
print(preds)

torch.Size([4])
tensor([-0.0693, -0.0982,  0.0142, -0.0833])


# Embeddings KFold

In [10]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

N_SPLITS = 5
EPOCHS = 100
BATCH_SIZE = 64
LR = 1e-3
SEED = 42

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

fold_rmse = []

for fold, (train_idx, val_idx) in enumerate(cv.split(X), start=1):
    X_train = X.iloc[train_idx]
    X_val = X.iloc[val_idx]

    y_train = y_log.iloc[train_idx]
    y_val = y_log.iloc[val_idx]
    
    cat_preprocessor = Pipeline([
        ("imputer", SimpleImputer(strategy="constant",fill_value="Missing")),
        ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
    ])
    
    X_train_cat = cat_preprocessor.fit_transform(X_train[cat_features])
    X_val_cat = cat_preprocessor.transform(X_val[cat_features])
    
    X_train_cat = X_train_cat.astype(np.int64) + 1
    X_val_cat = X_val_cat.astype(np.int64) + 1
    
    encoder = cat_preprocessor.named_steps['encoder']

    cardinalities = [
        len(categories) + 1
        for categories in encoder.categories_
    ]    
    
    num_preprocessor = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])
    
    X_train_num = num_preprocessor.fit_transform(X_train[num_features])
    X_val_num = num_preprocessor.transform(X_val[num_features])
    
    model = HpEmbeddingMLP(
        num_numeric=X_train_num.shape[1], 
        cardinalities=cardinalities
    ).to(device)
    
    X_train_cat = torch.tensor(X_train_cat, dtype=torch.long)
    X_train_num = torch.tensor(X_train_num, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32)
    X_val_cat = torch.tensor(X_val_cat, dtype=torch.long)
    X_val_num = torch.tensor(X_val_num, dtype=torch.float32)
    y_val_tensor = torch.tensor(y_val.values, dtype=torch.float32)
    
    train_dataset = TensorDataset(X_train_num, X_train_cat, y_train_tensor)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    
    loss_fn = nn.MSELoss()
    
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    
    for epoch in range(EPOCHS):
        model.train()
        
        for x_num_batch, x_cat_batch, y_batch in train_loader:
            x_num_batch = x_num_batch.to(device)
            x_cat_batch = x_cat_batch.to(device)
            y_batch = y_batch.to(device)
            
            optimizer.zero_grad()
            
            preds = model(x_num_batch, x_cat_batch)
            
            loss = loss_fn(preds, y_batch)
            
            loss.backward()
            optimizer.step()
            
    model.eval()
    
    with torch.no_grad():
        val_preds = model(
            X_val_num.to(device), 
            X_val_cat.to(device)
        )
        
        val_mse = loss_fn(val_preds, y_val_tensor.to(device))
        
        val_rmse = torch.sqrt(val_mse).item()
        
        fold_rmse.append(val_rmse)

        print(
            f"Fold {fold}: "
            f"RMSE = {val_rmse:.6f}"
        )
        
    
fold_rmse = np.array(fold_rmse)

print("\nFold RMSE:", fold_rmse)
print("Mean RMSE:", fold_rmse.mean())
print("Std RMSE:", fold_rmse.std())
    
    
    
    
    

Fold 1: RMSE = 0.179627
Fold 2: RMSE = 0.180661
Fold 3: RMSE = 0.216751
Fold 4: RMSE = 0.183425
Fold 5: RMSE = 0.200035

Fold RMSE: [0.17962702 0.18066093 0.21675071 0.18342549 0.20003517]
Mean RMSE: 0.19209986329078674
Std RMSE: 0.014368710770189823
